# Phase 8 — Statistical Validation of Assumptions
## OTT Platform Viewing Patterns Analysis

**Significance level:** α = 0.05

This notebook uses the cleaned datasets from Phases 2–7 and validates the Phase 3 hypotheses. It includes probability analysis, a t-test, a z-test, supporting statistical tests, decisions and business interpretation.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, math
from scipy import stats

DATA_DIR = Path('.')
view = pd.read_excel(DATA_DIR/'viewing_activity_cleaned.xlsx')
sub = pd.read_excel(DATA_DIR/'subscription_retention_cleaned.xlsx')
ratings = pd.read_excel(DATA_DIR/'ratings_feedback_cleaned.xlsx')
content = pd.read_excel(DATA_DIR/'content_library_cleaned.xlsx')
users = pd.read_excel(DATA_DIR/'user_profile_cleaned.xlsx')

print('Viewing:', view.shape)
print('Subscription:', sub.shape)
print('Ratings:', ratings.shape)
print('Content:', content.shape)
print('Users:', users.shape)

## 1. Probability Analysis

In [ ]:
sub['Churn_Flag'] = pd.to_numeric(sub['Churn_Flag'], errors='coerce')
total = len(sub)
churned = int(sub['Churn_Flag'].sum())
renewed = int((sub['Renewal_Status'].astype(str).str.strip().str.lower()=='renewed').sum())
print(f'P(Churn) = {churned}/{total} = {churned/total:.3%}')
print(f'P(Renewal) = {renewed}/{total} = {renewed/total:.3%}')
for plan in ['Free Trial','Basic','Premium']:
    g=sub[sub['Subscription_Type']==plan]
    print(f'P(Churn | {plan}) = {g.Churn_Flag.mean():.3%}')

## 2. H1 — Movies vs TV Shows Completion
**H0:** No significant difference in average completion.

**H1:** Significant difference in average completion.

**Test:** Welch independent-samples t-test.

In [ ]:
view['Content_Key']=pd.to_numeric(view['Content_ID'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
content['Content_Key']=pd.to_numeric(content['show_id'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
view2=view.merge(content[['Content_Key','type','Content_Age_Years']], on='Content_Key', how='left')
movie=view2.loc[view2['type']=='Movie','Completion_Percentage'].dropna()
tv=view2.loc[view2['type']=='TV Show','Completion_Percentage'].dropna()
res=stats.ttest_ind(movie,tv,equal_var=False)
print(f'Movie mean = {movie.mean():.2f}% | n={len(movie)}')
print(f'TV Show mean = {tv.mean():.2f}% | n={len(tv)}')
print(f't = {res.statistic:.4f} | p = {res.pvalue:.4e}')
print('Decision:', 'Reject H0' if res.pvalue<0.05 else 'Fail to reject H0')

## 3. H4 — Monthly Fee vs Churn: Two-Proportion Z-Test
The fee structure contains ₹0 trial subscriptions and paid subscriptions. The z-test compares churn proportions between these two groups.

In [ ]:
free=sub[sub['Monthly_Fee']==0]['Churn_Flag']
paid=sub[sub['Monthly_Fee']>0]['Churn_Flag']
x1,n1=int(free.sum()),len(free); x2,n2=int(paid.sum()),len(paid)
p1,p2=x1/n1,x2/n2
pp=(x1+x2)/(n1+n2)
se=math.sqrt(pp*(1-pp)*(1/n1+1/n2))
z=(p1-p2)/se
p=2*stats.norm.sf(abs(z))
print(f'Zero-fee churn = {x1}/{n1} = {p1:.2%}')
print(f'Paid churn = {x2}/{n2} = {p2:.2%}')
print(f'z = {z:.4f} | p = {p:.4e}')
print('Decision:', 'Reject H0' if p<0.05 else 'Fail to reject H0')

## 4. Supporting Validation of H2, H3, H5 and H6

In [ ]:
# H2
h2=view2[['Content_Age_Years','Completion_Percentage']].dropna()
r,p=stats.pearsonr(h2['Content_Age_Years'],h2['Completion_Percentage'])
print(f'H2 Pearson r={r:.4f}, p={p:.4f}')

# H3
us=users.merge(sub,on='User_ID',how='inner')
ct=pd.crosstab(us['Engagement_Level'],us['Churn_Flag'])
chi,p,dof,exp=stats.chi2_contingency(ct)
print(f'H3 Chi-square={chi:.4f}, p={p:.4f}')

# H5
vr=view.groupby('Content_ID')['Completion_Percentage'].mean().rename('Avg_Completion')
rr=ratings.groupby('Content_ID')['Rating'].mean().rename('Avg_Rating')
h5=pd.concat([vr,rr],axis=1).dropna()
r,p=stats.pearsonr(h5['Avg_Rating'],h5['Avg_Completion'])
print(f'H5 Pearson r={r:.4f}, p={p:.4f}')

# H6
rw=view.assign(Rewatch=(view['Rewatched_Flag'].astype(str).str.upper()=='Y').astype(int)).groupby('Content_ID')['Rewatch'].mean().rename('Rewatch_Rate')
h6=pd.concat([rr,rw],axis=1).dropna()
r,p=stats.pearsonr(h6['Avg_Rating'],h6['Rewatch_Rate'])
print(f'H6 Pearson r={r:.4f}, p={p:.4f}')

## 5. Final Interpretation
- H1: Reject H0 — Movies and TV Shows differ significantly in average completion.
- H4: Reject H0 — zero-fee trial and paid subscriptions have significantly different churn proportions.
- H2, H3, H5 and H6: the calculated p-values are above 0.05, so the null hypotheses are not rejected.

Do not claim causation from these tests; report association/difference only.